In [ ]:
## https://www.mdpi.com/2072-6694/14/15/3615
# Patients were defined as responders when they achieved PR or CR and as non-responders when categorized as PD or SD.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from lifelines import CoxPHFitter
from datetime import datetime, timedelta
import copy
import torch
def isin(name, seq):
    """Test if 'name' is a subsequence of the string 'seq'."""

    res = False
    return (name.lower() in seq.lower())

def isinlist(name,ls):
    """Test if 'name' is part of the list 'ls'."""
    res = False
    for name2 in ls:
        res = res or (isin(name, name2))
    return res

In [ ]:
import sys
sys.path.append('/data/users/quentin/final_package/')
import scClone2DR as sccdr
COHORT = 'MEL'
mode_features = 'metacells_gene_scatrex'
if COHORT=='MEL':
    path_rna = '/data/users/04_share_reanalysis_results/melanoma_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
    path_fastdrug = '/data/users/04_share_reanalysis_results/02_melanoma/MEL_CNN_abundances_no_plate_effect_correction.csv'
    concentration_DMSO = '100'
    concentration_drug = '5'
model = sccdr.models.scClone2DR(path_fastdrug=path_fastdrug, path_rna=path_rna)
data_ref = model.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)

In [ ]:
# Computing scores from Fast Drug data
sample_names = model.sample_names
sample2drug2FDratio_well = {sample:{} for sample in sample_names}
sample2FDproportions = {sample:np.array([0.5,0.5]) for sample in sample_names}
sample2drug2FDproba = {sample:{} for sample in sample_names}
for id_sample, sample in enumerate(sample_names):
    nb_wells = torch.sum(data_ref['n0_c'][:,id_sample]>0.5)
    sample2FDproportions[sample][0] = (torch.mean(data_ref['n0_c'][:nb_wells,id_sample]/data_ref['n_c'][:nb_wells,id_sample])).item()
    sample2FDproportions[sample][1] = 1-sample2FDproportions[sample][0]
    for d, drug in enumerate(model.FD.selected_drugs):
        if data_ref['n_r'][0,d,id_sample]>0.5:
            nb_wells = torch.sum(data_ref['n0_r'][:,d,id_sample]>0.5)
            props_well = np.array([0.5,0.5])
            props_well[0] = torch.mean(data_ref['n0_r'][:nb_wells,d,id_sample]/data_ref['n_r'][:nb_wells,d,id_sample])
            props_well[1] = 1-props_well[0]
            RBF = (props_well[1]/props_well[0]) * (sample2FDproportions[sample][0]/sample2FDproportions[sample][1])
            sample2drug2FDproba[sample][drug] = RBF.item()

In [ ]:
############ Loading the parameters learned using the becnhmark methods (FM and NN)
############ The files below were saved running the notebook "get_scores_from_benchmark_models.ipynb" in the ./data/ folder

import h5py
filename = "/data/users/quentin/final_package/experiments/survival_analysis//data/parameters_FM.h5"
with h5py.File(filename, "r") as f:
    # Print all root level object names (aka keys) 
    # these can be group or dataset names 
    print("Keys: %s" % f.keys())
    # get first object name/key; may or may NOT be a group
    FMsamples = f['pi'].attrs['samples']
    FMdrugs = f['pi'].attrs['drugs']
    pi_FM = f['pi'][:]
    props_FM = f['proportions'][:]
    
    
drugsample2probas_FM = {}
sample2proportions_FM = {}
for id_patient, patient in enumerate(FMsamples):
    sample2proportions_FM[patient] = props_FM[id_patient,:]
    for id_drug, drug in enumerate(FMdrugs):
        drugsample2probas_FM[(drug, patient)] = pi_FM[id_drug,:,id_patient]
        
filename = "/data/users/quentin/final_package/experiments/survival_analysis//data/parameters_NN.h5"
with h5py.File(filename, "r") as f:
    # Print all root level object names (aka keys) 
    # these can be group or dataset names 
    print("Keys: %s" % f.keys())
    # get first object name/key; may or may NOT be a group
    NNsamples = f['pi'].attrs['samples']
    NNdrugs = f['pi'].attrs['drugs']
    pi_NN = f['pi'][:]
    props_NN = f['proportions'][:]
    
    
drugsample2probas_NN = {}
sample2proportions_NN = {}
for id_patient, patient in enumerate(NNsamples):
    sample2proportions_NN[patient] = props_NN[id_patient,:]
    for id_drug, drug in enumerate(NNdrugs):
        drugsample2probas_NN[(drug, patient)] = pi_NN[id_drug,:,id_patient]

In [ ]:
drugsample2probas_NN

# 1) BUILD MATRIX FOR TREATMENT DURATION

# Filtering samples: we keep those considered in the WP1

In [ ]:
df = pd.read_excel('/data/users/quentin/final_package/experiments/survival_analysis/data/Melanoma_clinical_data_jack_may7.xlsx', '20211001_Melanoma_clinical_data')
feature_names = ['tupro_id', 'age','sex','brain_metastasis',
                 'treatment_after',"stage", "subtype",
                 'tumor_content_in_percent', 'metastasis_location', 'biopsy_date']
responses = ['response_{0}_months'.format(int(3*i)) for i in range(1,10)] 
response2time = {'response_{0}_months'.format(int(3*i)):int(3*i) for i in range(1,10)}
df = df[feature_names + responses]
df.head()

In [ ]:
# DATA FROM WORK PACKAGE 1
import h5py
directory = '/data/users/quentin/final_package/experiments/survival_analysis//data/'
import pickle
if False:
    # Open the HDF5 file in read mode
    with h5py.File(directory+'survival_probabilities_0.h5', 'r') as f:
        # Access the dataset
        dset = f['survival_probabilities_posterior_mean']
        probas = dset[:]
        dset = f['survival_probabilities_posterior_std']
        annotations = {dim: dset.attrs[dim] for dim in dset.attrs}

else:


    dire = '/data/users/quentin/final_package/experiments/paper_results/melanoma/hallmarks_scatrex/'
    with h5py.File(dire+'local_importance.h5', 'r') as f:
        # Access the dataset
        print(list(f['local_importance_mean'].shape))
        print(f['local_importance_mean'].attrs.keys())
        drugs = f['local_importance_mean'].attrs['dim1_drugs']
        samples = f['local_importance_mean'].attrs['dim3_samples']
        
        with open(dire+'posterior_mean_params.pkl', 'rb') as handle:
            posterior_mean_params = pickle.load(handle)
            probas = posterior_mean_params['pi']
            probas = np.array(probas)

try:
    sample_names = [''.join(chr(i) for i in el) for el in samples]
except:
    sample_names = samples
# List of all drugs we have in the Work Package 1
#np.sort(drugs)
probas.shape

In [ ]:
# dir_save = '/data/users/quentin/Paper_WP1/expes_paper/real_random_splits/LDVAE/save_all_patients/'

# nuh = np.load(dir_save+'nu_healthy_drug_{0}.npy'.format(0))
# meannuh = np.sum(nuh*(nuh!=0.5))/np.sum(nuh!=0.5)
# meannuh

In [ ]:
# Filtering samples on clinical data: We keep only the sample we used to train the model for the 1st work package
df = df.loc[df['tupro_id'].apply(lambda x: x in sample_names)]
df = df.set_index('tupro_id')
#df = df.loc[df2_res.index]
df_res = df[responses]
print(len(df_res))

In [ ]:
df_res.head()

# Possible responses

In [ ]:
# all possible responses
acronyms = {'PD': 'progressive disease', 
            'PR': 'partial response',
            'CR': 'complete response',
            'MR': 'minimal response',
            'SD': 'stable disease',
            'died': 'died',
            'relapse free': 'relapse free' ,
            'no assessment possible': 'no assessment possible'
            # Defined only for patients achieving CR: 
            # measured from the date of achievement of a remission until the date of relapse from any cause
           }
np.unique(df[responses[:-1]].astype(str).to_numpy().reshape(-1))

Complete Response (CR): Disappearance of all target lesions.
Any pathological lymph nodes (whether target or
non-target) must have reduction in short axis to
<10 mm.

Partial Response (PR): At least a 30% decrease in the sum of
diameters of target lesions, taking as reference the
baseline sum diameters.

Progressive Disease (PD): At least a 20% increase in the sum
of diameters of target lesions, taking as reference
the smallest sum on study (this includes the baseline
sum if that is the smallest on study). In addition to
the relative increase of 20%, the sum must also demonstrate an absolute increase of at least 5 mm. (Note:
the appearance of one or more new lesions is also
considered progression).

Stable Disease (SD): Neither sufficient shrinkage to qualify for
PR nor sufficient increase to qualify for PD, taking as
reference the smallest sum diameters while on study

# Drugs and classes of drugs

### 1) Immunotherapy

In [ ]:
# List of drugs (and group of drugs) related to immunotherapy that has been used after biopsy in the clinical data  
missing_drugs = ['Nivolumab', 'Ipilimumab', 'Spartalizumab',  'Pembrolizumab', 'anti-LAG3',
          'anti-CTLA-4' , 'anti-PD-1', 'Atezolizumab']

In [ ]:
class2immuno = {'anti-CTLA-4': ['Ipilimumab'],
    'anti-PD-1': ['Nivolumab', 'Pembrolizumab', 'PDR001', 'Spartalizumab'],
    'anti-LAG3': ['Relatlimab', 'LAG525'],
    'STING agonist': ['MIW815'],
    'anti-MCSF': ['MCS110'],
    'TVEC': ['TVEC'],
    'IDOi': ['BMS-986205'],
    'anti-PD-L1': ['Durvalumab', 'Atezolizumab', 'Avelumab'],
    'anti-IL1ß': ['Canakinumab', 'ACZ885']
                      }
immuno2class = {}
immuno_class_and_drugs = []
for clas,therapies in class2immuno.items():
    immuno_class_and_drugs.append(clas)
    for therapy in therapies:
        immuno2class[therapy] = clas
        immuno_class_and_drugs.append(therapy)

In [ ]:
# Investigating immunotherapy in the clinical data
other_drugs_clinical = []
immunoclasses_in_clinical = []
missing_classes = []
for i, sample in enumerate(df.index):
    for drug in missing_drugs:
        if isin(drug, df.loc[sample]['treatment_after']):
            other_drugs_clinical.append(drug)
            try:
                immunoclasses_in_clinical.append(immuno2class[drug])
            except:
                missing_classes.append(drug)
other_drugs_clinical = list(set(other_drugs_clinical))
missing_classes = list(set(missing_classes))
immunoclasses_in_clinical = list(set(immunoclasses_in_clinical))
print('immuno_in_clinical: ', len(other_drugs_clinical))
print('missing_classes: ', missing_classes)
print('classes_in_clinical: ', immunoclasses_in_clinical)

immunoclasses_in_clinical = list(set(immunoclasses_in_clinical + missing_classes))
print('Final classes_in_clinical', immunoclasses_in_clinical)

### 2) Targeted therapy

In [ ]:
# https://www.mycancergenome.org/content/page/overview-of-targeted-therapies-for-cancer/
class2targeted_therapy = {'BRAFi':['LGX818','Vemurafenib', 'Dabrafenib', 'Zelboraf', 'Taflinar', 'Encorafenib', 'Regorafenib'],
    'MEKi': ['Trametinib','Mekinist', 'Cobimetinib', 'Cotellic', 'MEK162', 'Binimetinib'],
    'panRAFi': ['LXH254'],
    'METi': ['Crizotinib', 'Cabozantinib', 'INC280', 'Capamatinib'],
    'PARPi': ['Olaparib', 'Niraparib'],
    'CDK4/6i':['LEE011', 'Ribociclib', 'Palbociclib'],
    'proteasome inhibitor': ['Carfilzomib', 'Ixazomib', 'Bortezomib'],
    'VEGFR2i': ['Lenvatinib'],
    'ABLi': ['Nilotinib'],
    'chemotherapies': ['Temozolomide', 'Carboplatin', 'Vindesine', 'Cisplatin', 'Taxol/Taxane', 'Dacarbazine', 'Paclitaxel']
                         }

targeted_therapy2class = {}
for clas,therapies in class2targeted_therapy.items():
    for therapy in therapies:
        targeted_therapy2class[therapy] = clas

In [ ]:
# Investigating targeted therapy in the clinical data
targeted_in_clinical = []
targetedclasses_in_clinical = []
missing_classes = []
for i, sample in enumerate(df.index):
    for drug in drugs:
        if isin(drug, df.loc[sample]['treatment_after']):
            targeted_in_clinical.append(drug)
            try:
                targetedclasses_in_clinical.append(targeted_therapy2class[drug])
            except:
                missing_classes.append(drug)
targeted_in_clinical = list(set(targeted_in_clinical))
missing_classes = list(set(missing_classes))
targetedclasses_in_clinical = list(set(targetedclasses_in_clinical))
print('targeted_in_clinical: ', len(targeted_in_clinical))
print('missing_classes: ', missing_classes)
print('classes_in_clinical: ', targetedclasses_in_clinical)

# Building the design matrix for the Cox PH model

#### Event: response $\in \{$Progressive disease, Stable disease$\}$.

#### Censored: response $\in \{$died, out of study$\}$.

In [ ]:
data = []
features = ['sampleID', 'age', 'sex', 'brain_metastasis', 'begin', 'end', 'duration',  'event_observed']
features += immunoclasses_in_clinical
features += targetedclasses_in_clinical
features += ['list_targeted']
feature2idx = {feature: i for i,feature in enumerate(features)}
for i, sample in enumerate(df.index):
    ref_row = []
    ref_row.append(sample)
    for feature in features[1:4]:
        ref_row.append(df.loc[sample][feature])
    treat = df.loc[sample]['treatment_after']
    blocks = treat.split(".")
    try:
        current_date = datetime.strptime(treat[:8], '%Y%m%d')
    except: 
        try:
            current_date = datetime.strptime(treat[:6], '%Y%m')
        except:
            print('Error', sample)
            continue
    
    for block in blocks:
        row = copy.deepcopy(ref_row)
        
        # Find two dates
        n = len(block)
        all_dates = []
        for j in range(n-8):           
            try:
                tempdate = datetime.strptime(block[j:j+8], '%Y%m%d')
                all_dates.append(tempdate)
            except:
                if block[j:j+7]=='ongoing':
                    print(sample, 'ongoing',all_dates)
                    all_dates.append(datetime.strptime('205001', '%Y%m'))
                elif block[j+6] in ['-',':'] and block[j-2:j]!='20':
                    try:
                        tempdate = datetime.strptime(block[j:j+6], '%Y%m')
                        all_dates.append(tempdate)
                    except:
                        pass
        if len(all_dates)==2 and ((all_dates[1]-max(current_date, all_dates[0])).days>4):
            duration = (all_dates[1]-max(current_date, all_dates[0])).days
            
            row.append(max(current_date, all_dates[0]))
            row.append(all_dates[1])
            
            ls_drugs = []
            ls_target = []
            for drug in drugs:
                if isin(drug,block):
                    ls_drugs.append(targeted_therapy2class[drug])
                    ls_target.append(drug)
            ls_other_drugs = []
            for drug in immuno_class_and_drugs:
                if isin(drug,block):
                    if isinlist(drug,immunoclasses_in_clinical):
                        ls_other_drugs.append(drug)
                    else:
                        ls_other_drugs.append(immuno2class[drug])
            ls_drugs = list(set(ls_drugs))
            ls_other_drugs = list(set(ls_other_drugs))
            
            response = []
            for res in acronyms.keys():
                if res in block:
                    response.append(res)

            row.append(duration)
            row.append(response)
            for immuno_class in immunoclasses_in_clinical:
                if isinlist(immuno_class,ls_other_drugs):
                    row.append(1)
                else:
                    row.append(0)
                
            for targeted_class in targetedclasses_in_clinical:
                if targeted_class in ls_drugs:
                    row.append(1)
                else:
                    row.append(0)
            row.append(ls_target)
            data.append(row)
        current_date = max([current_date]+all_dates)

In [ ]:
X = pd.DataFrame(data=data, columns=features)
#X = X.set_index('sampleID')
X['sex'] = X['sex'].apply(lambda x: 1*(x=='female'))
X['age'] = X['age'].apply(lambda x: int(x[0])*10+5)
X['brain_metastasis'] = X['brain_metastasis'].apply(lambda x: 1*(x=='yes'))
#X = X.loc[:, (X != X.iloc[0]).any()] # remove constant columns

In [ ]:
X['immunotherapies'] = X.loc[:,immunoclasses_in_clinical].apply(lambda x: 1*(np.sum(x.to_numpy().astype(int))>=1),axis=1)
X['targeted'] = X.loc[:,targetedclasses_in_clinical].apply(lambda x: 1*(np.sum(x.to_numpy().astype(int))>=1),axis=1)
X

In [ ]:
X.loc[X['sampleID']=='MYJILAS']

In [ ]:
X

# 2) BUILD FINAL DATAFRAME

In [ ]:
import pickle
with open('/data/users/quentin/final_package/experiments/survival_analysis/data/params_svi.pkl', 'rb') as handle:
    params_svi = pickle.load(handle)
    
proportions = params_svi['proportions']
print(proportions.shape)
Kmax = proportions.shape[1]

In [ ]:

sample2time2drugs = {}
features_final = ['tupro_id', 'age','sex','brain_metastasis','stage', 'subtype', 'biopsy_date', 'metastasis_location', 'state', 'prev_state', 'days']
for thera in ['immunotherapies', 'chemotherapies', 'targeted']:
    features_final.append(thera+'_tot_after_biopsy')
    features_final.append(thera+'_tot_all_time')
    features_final.append(thera+'_tot_prev_state')
    features_final.append(thera+'_lag')
    
for drug in drugs:
    features_final.append(drug+'_tot_after_biopsy')
    features_final.append(drug+'_tot_prev_state')
    features_final.append(drug+'_lag')


for model_used in ['WP1', 'NN', 'FM']:
    ########## RESISTANT WP1
    features_final.append('resistant_{0}_NM'.format(model_used)) 
    features_final.append('resistant_{0}_T'.format(model_used))
    features_final.append('resistant_{0}_NM_norm'.format(model_used))
    features_final.append('resistant_{0}_T_norm'.format(model_used))

    features_final.append('resistant_{0}_NM_long'.format(model_used)) 
    features_final.append('resistant_{0}_T_long'.format(model_used))
    features_final.append('resistant_{0}_NM_norm_long'.format(model_used))
    features_final.append('resistant_{0}_T_norm_long'.format(model_used))

    ########### FRACTION WP1
    features_final.append('fraction_{0}'.format(model_used))
    features_final.append('fraction_{0}_NM_norm'.format(model_used))
    features_final.append('fraction_{0}_T_norm'.format(model_used))

    features_final.append('fraction_{0}_long'.format(model_used)) # taking into account both time evolution and subclones' proportions
    features_final.append('fraction_{0}_NM_norm_long'.format(model_used)) # taking into account both time evolution and subclones' proportions
    features_final.append('fraction_{0}_T_norm_long'.format(model_used)) # taking into account both time evolution and subclones' proportions

    ########## SCORE WP1
    features_final.append('score_{0}_NM'.format(model_used)) 
    features_final.append('score_{0}_T'.format(model_used))
    features_final.append('score_{0}_NM_norm'.format(model_used))
    features_final.append('score_{0}_T_norm'.format(model_used))

    features_final.append('score_{0}_NM_long'.format(model_used)) 
    features_final.append('score_{0}_T_long'.format(model_used))
    features_final.append('score_{0}_NM_norm_long'.format(model_used))
    features_final.append('score_{0}_T_norm_long'.format(model_used))



######### FRACTION FAST DRUG
features_final.append('fraction_FD')
features_final.append('fraction_FD_NM_norm')
features_final.append('fraction_FD_T_norm')

features_final.append('fraction_FD_long') # taking into account both time evolution and subclones' proportions
features_final.append('fraction_FD_NM_norm_long') # taking into account both time evolution and subclones' proportions
features_final.append('fraction_FD_T_norm_long') # taking into account both time evolution and subclones' proportions


######## SCORE FAST DRUG
features_final.append('score_FD_NM')
features_final.append('score_FD_T')
features_final.append('score_FD_NM_norm')
features_final.append('score_FD_T_norm')

features_final.append('score_FD_NM_long')
features_final.append('score_FD_T_long')
features_final.append('score_FD_NM_norm_long')
features_final.append('score_FD_T_norm_long')

features_final.append('log_props_control_WP1_NM')
features_final.append('log_props_control_WP1_T')
features_final.append('log_props_WP1_NM')
features_final.append('log_props_WP1_T')
features_final.append('log_props_WP1_NM_long')
features_final.append('log_props_WP1_T_long')


features_final.append('relative_tumor_evolution_WP1')
features_final.append('relative_tumor_evolution_NN')
features_final.append('relative_tumor_evolution_FM')
features_final.append('relative_tumor_evolution_FD')


features_final.append('relative_tumor_evolution_from_base_WP1')
features_final.append('relative_tumor_evolution_from_base_NN')
features_final.append('relative_tumor_evolution_from_base_FM')
features_final.append('relative_tumor_evolution_from_base_FD')


def compute_stats_resistant(props, probas):
    res = []
    if len(probas)!=1:
        idx_resistant = np.argmax(probas[1:])+1
        r_nm = np.clip(props[0]*probas[0]/(props[idx_resistant]*probas[idx_resistant]), a_min=0, a_max=15)
        res.append(r_nm)
        r_t = 1/r_nm
        res.append(r_t)
        r_nm_norm = np.clip(probas[0]/(probas[idx_resistant]), a_min=0, a_max=15)
        res.append(r_nm_norm)
        r_t_norm = 1/r_nm_norm
        res.append(r_t_norm)
    else:
        r_nm = 15
        res.append(r_nm)
        r_t = 1/r_nm
        res.append(r_t)
        r_nm_norm = 15
        res.append(r_nm_norm)
        r_t_norm = 1/r_nm_norm
        res.append(r_t_norm)
    return res
    

def compute_stats_fraction(props, probas, FD=False):
    res = []
    if not(FD):
        f = np.clip(props[0] * probas[0] /np.sum(props*probas), a_min=0, a_max=15)
        res.append(f)
    f_nm_norm = np.clip(probas[0] /np.sum(props*probas), a_min=0, a_max=15)
    res.append(f_nm_norm)
    f_t_norm = np.clip(np.sum(props[1:]*probas[1:]) / (np.sum(props*probas)*np.sum(props[1:])), a_min=0, a_max=15)
    res.append(f_t_norm)
    return res

                       
def compute_stats_score(props, probas, FD=False):
    res = []
    if (np.sum(props[1:]) <1e-4):
        s_nm = 15
        s_nm_norm = 15
    else:
        s_nm = props[0]*probas[0] / np.sum(props[1:] * probas[1:]) 
        s_nm_norm =  probas[0] * np.sum(props[1:]) / np.sum(props[1:] * probas[1:]) 

    if not(FD):
        res.append(s_nm)
        s_t = np.sum(props[1:] * probas[1:])  / (props[0]*probas[0]) 
        res.append(s_t)
    
    res.append(s_nm_norm)
    s_t_norm = np.sum(props[1:] * probas[1:])  / (probas[0] * np.sum(props[1:]))
    res.append(s_t_norm)
    return res
  
import copy
data = []
for id_sample, sample in enumerate(sample_names):
    sample2time2drugs[sample] = {}
    ## for the time evolution of the tumor
    nb_tumor_sub = np.sum(~pd.isnull(probas[0,:,id_sample]))
    evolve_sizes = proportions[id_sample,:nb_tumor_sub]

    prevstate = np.nan
    
    X_treat = X.loc[X['sampleID']==sample]
    
    # long_term_evolution: we accumulate the effect of the targeted drug along time
    allprobas_FD_long = np.ones(2)
    allprobas_FM_long = np.ones(nb_tumor_sub)
    allprobas_NN_long = np.ones(nb_tumor_sub)
    allprobas_long = np.ones(nb_tumor_sub)
    
    
    prev_tumor_content_NN = np.sum(sample2proportions_NN[sample][1:nb_tumor_sub])
    prev_tumor_content_FM = np.sum(sample2proportions_FM[sample][1:nb_tumor_sub])
    prev_tumor_content_FD = np.sum(sample2FDproportions[sample][1:])
    prev_tumor_content_WP1 = np.sum(proportions[id_sample,1:])
    prev_tumor_contents = {'WP1': prev_tumor_content_WP1, 
                           'NN': prev_tumor_content_NN,
                           'FM': prev_tumor_content_FM,
                           'FD': prev_tumor_content_FD
                          }
    current_tumor_contents = copy.deepcopy(prev_tumor_contents)
    ini_tumor_contents = copy.deepcopy(prev_tumor_contents)

    for i_res, resp in enumerate(responses):
        sample2time2drugs[sample][i_res] = []
        row = [sample]
        for el in ['age','sex','brain_metastasis','stage', 'subtype', 'biopsy_date', 'metastasis_location']:
            row.append(df.loc[sample][el])
        row.append(df_res.loc[sample, resp])
        row.append(prevstate)
        prevstate = df_res.loc[sample, resp]
        row.append(response2time[resp]*30)
        
        biopsy_date = datetime.strptime(str(df.loc[sample]['biopsy_date'])[:8], '%Y%m%d')
        
        date_begin = biopsy_date +  timedelta(days=(response2time[resp]-3)*30)
        date_end = biopsy_date +  timedelta(days=(response2time[resp])*30)
        
        # Identify if a class of treatment was taken and the last time it was
        for thera in ['immunotherapies', 'chemotherapies', 'targeted']:
            after_bio = timedelta(days=0)
            alltime = timedelta(days=0)
            prestate = timedelta(days=0)
            
            lastdate4lag = biopsy_date #datetime.strptime('19000101', '%Y%m%d')
            for index, rowt in X_treat.iterrows():
                assert (rowt['begin']<rowt['end'])
                if rowt['begin']<=date_end:
                    if rowt[thera]==1:
                        alltime += min([rowt['end'],date_end])-rowt['begin']
                        after_bio += min([rowt['end'],date_end])-max([rowt['begin'],biopsy_date])
                        after_bio = max([timedelta(days=0), after_bio])
                        prestate +=  min([rowt['end'],date_end])-max([rowt['begin'],date_begin])
                        prestate = max([timedelta(days=0), prestate])

                        if rowt['end']<=date_begin:
                            lastdate4lag = max([lastdate4lag, rowt['end']])
                        else:
                            lastdate4lag = date_end
                                                    
                
            # tot_after_biospy
            row.append(after_bio.days)
            # tot_all_time
            row.append(alltime.days)
            # tot_prev_state
            row.append(prestate.days)
            # lag
            if alltime.days==0:
                row.append(0)
            else:
                row.append((date_end - lastdate4lag).days)
                
        # Identify if a targeted treatment was taken and the last time it was
        for drug in drugs:
            after_bio = timedelta(days=0)
            prestate = timedelta(days=0)
            
            lastdate4lag = biopsy_date #datetime.strptime('19000101', '%Y%m%d')
            for index, rowt in X_treat.iterrows():
                assert (rowt['begin']<rowt['end'])
                if rowt['begin']<=date_end:
                    if drug in rowt['list_targeted']:
                        after_bio += min([rowt['end'],date_end])-max([rowt['begin'],biopsy_date])
                        after_bio = max([timedelta(days=0), after_bio])
                        prestate +=  min([rowt['end'],date_end])-max([rowt['begin'],date_begin])
                        prestate = max([timedelta(days=0), prestate])

                        if rowt['end']<=date_begin:
                            lastdate4lag = max([lastdate4lag, rowt['end']])
                        else:
                            lastdate4lag = date_end
                                                    
            # tot_after_biospy
            row.append(after_bio.days)
            # tot_prev_state
            row.append(prestate.days)
            # lag
            if alltime.days==0:
                row.append(0)
            else:
                row.append((date_end - lastdate4lag).days)
                
        # Add survival probabilities
        allprobas = np.ones(nb_tumor_sub)
        allprobas_FD = np.ones(2)

        allprobas_FM = np.ones(nb_tumor_sub)
        allprobas_NN = np.ones(nb_tumor_sub)

        for id_drug, drug in enumerate(drugs):
            for index, rowt in X_treat.iterrows():
                # check if the drug was taken during the last 
                if (rowt['begin']<=date_end) and (rowt['end']>=date_begin):
                    if drug in rowt['list_targeted']:
                        
                        sample2time2drugs[sample][i_res].append(drug)
                        # short term evolution
                        allprobas *= probas[id_drug,:nb_tumor_sub,id_sample]
                        allprobas_FM *= drugsample2probas_FM[(drug,sample)][:nb_tumor_sub]
                        allprobas_NN *= drugsample2probas_NN[(drug,sample)][:nb_tumor_sub]
                        allprobas_FD[1] *= sample2drug2FDproba[sample].get(drug, 1)

                        
                        # long term evolution
                        allprobas_long *= probas[id_drug,:nb_tumor_sub,id_sample]
                        allprobas_FM_long *= drugsample2probas_FM[(drug,sample)][:nb_tumor_sub]
                        allprobas_NN_long *= drugsample2probas_NN[(drug,sample)][:nb_tumor_sub]
                        allprobas_FD_long[1] *= sample2drug2FDproba[sample].get(drug, 1)

                        
        propsNN = sample2proportions_NN[sample][0:nb_tumor_sub]
        propsFM = sample2proportions_FM[sample][0:nb_tumor_sub]
        model2proportions = {"WP1": proportions[id_sample,:], "NN": propsNN, "FM": propsFM, "FD": sample2FDproportions[sample] }
        model2probas = {"WP1": allprobas, "NN": allprobas_NN, "FM": allprobas_FM, 'FD': allprobas_FD}
        model2probas_long = {"WP1": allprobas_long, "NN": allprobas_NN_long, "FM": allprobas_FM_long, "FD": allprobas_FD_long}

        for model_used in ['WP1', 'NN', 'FM', 'FD']:
            if model_used != 'FD':
                res = compute_stats_resistant(model2proportions[model_used][:nb_tumor_sub], model2probas[model_used][:nb_tumor_sub]**4)
                row += res
                res = compute_stats_resistant(model2proportions[model_used][:nb_tumor_sub], model2probas_long[model_used][:nb_tumor_sub]**4)
                row += res
            
            res = compute_stats_fraction(model2proportions[model_used][:nb_tumor_sub], model2probas[model_used][:nb_tumor_sub])
            row += res
            res = compute_stats_fraction(model2proportions[model_used][:nb_tumor_sub], model2probas_long[model_used][:nb_tumor_sub])
            row += res
            
            res = compute_stats_score(model2proportions[model_used][:nb_tumor_sub], model2probas[model_used][:nb_tumor_sub])
            row += res
            res = compute_stats_score(model2proportions[model_used][:nb_tumor_sub], model2probas_long[model_used][:nb_tumor_sub])
            row += res
            
            
        row.append(np.clip(np.log(proportions[id_sample,0]), a_min=-10, a_max=10))
        row.append(np.clip(np.log(1-proportions[id_sample,0]), a_min=-10, a_max=10))
        frac_NM = compute_stats_fraction(model2proportions['WP1'][:nb_tumor_sub], model2probas['WP1'][:nb_tumor_sub])[0]
        row.append(np.clip(np.log(frac_NM), a_min=-10, a_max=10))
        row.append(np.clip(np.log(1-frac_NM), a_min=-10, a_max=10))
        frac_NM_long = compute_stats_fraction(model2proportions['WP1'][:nb_tumor_sub], model2probas_long['WP1'][:nb_tumor_sub])[0]
        row.append(np.clip(np.log(frac_NM_long), a_min=-10, a_max=10))
        row.append(np.clip(np.log(1-frac_NM_long), a_min=-10, a_max=10))
        
        
        # relative tumor evolution
        for model_used in ['WP1', 'NN', 'FM', 'FD']:
            current_tumor_contents[model_used] = np.sum(model2probas_long[model_used][1:nb_tumor_sub] * model2proportions[model_used][1:nb_tumor_sub])
            current_tumor_contents[model_used] /= np.sum(model2probas_long[model_used][0:nb_tumor_sub] * model2proportions[model_used][0:nb_tumor_sub])
            import math
            rel_diff = (current_tumor_contents[model_used]-prev_tumor_contents[model_used])/prev_tumor_contents[model_used]
            if prev_tumor_contents[model_used]<1e-8:
                prev_tumor_contents[model_used] = 1e-6
                rel_diff = (current_tumor_contents[model_used]-prev_tumor_contents[model_used])/prev_tumor_contents[model_used]
            if math.isnan(rel_diff):
                print(current_tumor_contents[model_used],prev_tumor_contents[model_used])
            row.append( np.clip(rel_diff, a_min=-50, a_max=50))
        prev_tumor_contents = copy.deepcopy(current_tumor_contents)

        # relative tumor evolution from baseline
        for model_used in ['WP1', 'NN', 'FM', 'FD']:
            current_tumor_contents[model_used] = np.sum(model2probas_long[model_used][1:nb_tumor_sub] * model2proportions[model_used][1:nb_tumor_sub])
            current_tumor_contents[model_used] /= np.sum(model2probas_long[model_used][0:nb_tumor_sub] * model2proportions[model_used][0:nb_tumor_sub])
            import math
            rel_diff = (current_tumor_contents[model_used]-ini_tumor_contents[model_used])/ini_tumor_contents[model_used]
            if ini_tumor_contents[model_used]<1e-8:
                ini_tumor_contents[model_used] = 1e-6
                rel_diff = (current_tumor_contents[model_used]-ini_tumor_contents[model_used])/ini_tumor_contents[model_used]
            if math.isnan(rel_diff):
                print(current_tumor_contents[model_used],ini_tumor_content[model_used])
            row.append( np.clip(rel_diff, a_min=-50, a_max=50))
        prev_tumor_contents = copy.deepcopy(current_tumor_contents)

        
        rest = df_res.loc[sample,responses[i_res:]].to_numpy()
        if (np.sum(pd.isnull(rest))==len(rest)):
            sample2time2drugs[sample].pop(i_res)
            break
        elif i_res!=0:
            data.append(row)
        else:
            sample2time2drugs[sample].pop(i_res)


        
        if (prevstate=='died'):
            break
            
with open('sample2time2drugs.pkl', 'wb') as handle:
    pickle.dump(sample2time2drugs, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
possible_response = {'relapse free':1, 'CR':2, 'PR':3, 'MR':3, 'SD':4, 'PD':5, 'died':6, 'no': np.nan,'xx': np.nan}
def rename_state(a):
    if pd.isnull(a):
        return a
    for key,val in possible_response.items():
        if key in a:
            return val
    assert False

In [ ]:
probas.shape

In [ ]:
df_final = pd.DataFrame(data, columns = features_final)
df_final['state'] = df_final['state'].apply(lambda x: rename_state(x))
df_final['prev_state'] = df_final['prev_state'].apply(lambda x: rename_state(x))
df_final['sex'] = df_final['sex'].apply(lambda x: 1*(x=='female'))
df_final['age'] = df_final['age'].apply(lambda x: int(x[0])*10+5)
df_final['brain_metastasis'] = df_final['brain_metastasis'].apply(lambda x: 1*(x=='yes'))
df_final

In [ ]:
# Filling NaN values: looking at them, one can notice that filling with a label=5 (PD) is the way to go
for sample in sample_names:
    dfs = df_final.loc[df_final['tupro_id']==sample]
    if np.sum(np.isnan(dfs['state']))>=1:
        plt.figure()
        plt.scatter([i for i in range(dfs.shape[0])], dfs['state'])
        plt.title(sample)
        plt.show()

df_final['state'] = df_final['state'].fillna(5)
df_final['state'] = df_final['state'].astype(int)
df_final['prev_state'] = df_final['prev_state'].fillna(5)
df_final['prev_state'] = df_final['prev_state'].astype(int)

In [ ]:
df_final.to_csv('ord_longi.csv')

In [ ]:
dfgraph = df_final[['tupro_id', 'prev_state', 'state', 'days']].copy()
dfgraph['days'] -= 30 * 3

rows = []

for sample_name in dfgraph['tupro_id'].unique():
    dftemp = dfgraph[dfgraph['tupro_id'] == sample_name]

    last_row = dftemp.iloc[-1]

    rows.append({
        'tupro_id': sample_name,
        'days': last_row['days'] + 30 * 3,
        'prev_state': last_row['state'],
        'state': ''
    })

dfgraph = pd.concat([dfgraph, pd.DataFrame(rows)], ignore_index=True)
dfgraph['months'] = dfgraph["days"] /30
dfgraph['months'] = dfgraph['months'].astype(int)

In [ ]:
import copy
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from matplotlib.colors import ListedColormap, BoundaryNorm

# ------------------------------------------------------------------
# 1. Copy dataframe
# ------------------------------------------------------------------
df2 = copy.deepcopy(dfgraph)

# ------------------------------------------------------------------
# 2. Define clinical states and ordering
# ------------------------------------------------------------------
possible_response2 = {
    'relapse free': 1,
    'complete response': 2,
    'partial response': 3,
    'stable disease': 4,
    'progressive disease': 5,
    'died': 6
}

group2name = {i: name for name, i in possible_response2.items()}

# ------------------------------------------------------------------
# 3. Encode states as ordered categorical → integer codes (0–5)
# ------------------------------------------------------------------
state_order = list(possible_response2.keys())

# Convert categorical states to numeric codes
df2['prev_state'] = df2['prev_state'].astype('category')
df2['state_code'] = df2['prev_state'].cat.codes
df2['state_code'] = df2['prev_state'].astype('category').cat.codes
state_categories = state_order  # for colorbar labels

# ------------------------------------------------------------------
# 4. Anonymize patient IDs while preserving order
# ------------------------------------------------------------------
unique_patients = df2['tupro_id'].unique()
anon_labels = [f'Patient {i+1}' for i in range(len(unique_patients))]
anon_map = dict(zip(unique_patients, anon_labels))

df2['anon_id'] = df2['tupro_id'].map(anon_map)
df2['anon_id'] = pd.Categorical(
    df2['anon_id'],
    categories=anon_labels,
    ordered=True
)

# ------------------------------------------------------------------
# 5. Pivot for heatmap
# ------------------------------------------------------------------
pivot_df = df2.pivot(
    index='anon_id',
    columns='months',
    values='state_code'
)

# ------------------------------------------------------------------
# 6. Define discrete colormap (EXACT colors per state)
# ------------------------------------------------------------------
colors = ['#8c510a', '#d8b365', '#f6e8c3', '#c7eae5', '#5ab4ac', '#01665e'][::-1]
cmap = ListedColormap(colors)
norm = BoundaryNorm(
    boundaries=range(len(colors) + 1),
    ncolors=len(colors)
)

# ------------------------------------------------------------------
# 7. Plot heatmap
# ------------------------------------------------------------------
plt.figure(figsize=(12, 8))
ax = sns.heatmap(
    pivot_df,
    cmap=cmap,
    norm=norm,
    cbar=True,
    linewidths=0.2,
    linecolor='white'
)

# ------------------------------------------------------------------
# 8. Correct colorbar ticks & labels (centered)
# ------------------------------------------------------------------
cbar = ax.collections[0].colorbar
cbar.set_ticks([i + 0.5 for i in range(len(state_categories))])
cbar.set_ticklabels(state_categories)



# ------------------------------------------------------------------
# 9. Labels and export
# ------------------------------------------------------------------
plt.xlabel('Time in months')
plt.ylabel('Patients')

# Increase axis label size
ax.set_xlabel('Time in months', fontsize=14)
ax.set_ylabel('Patients', fontsize=14)

# Increase tick label sizes
ax.tick_params(axis='x', labelsize=13)
ax.tick_params(axis='y', labelsize=13)
ax.set_yticks([])

# Increase colorbar tick label size
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=15)

plt.tight_layout()
plt.savefig('clinical_data_mel.png', dpi=250, bbox_inches='tight')
plt.show()


# Proportions of the different responses along time

In [ ]:
sample_names_df_final = np.unique(df_final['tupro_id'])
all_y = []
for sample in sample_names_df_final:
    df_p = df_final.loc[df_final['tupro_id']==sample]
    df_p = df_p.sort_values(['days']).reset_index(drop=True)
    y = [df_p.iloc[0]['prev_state']]
    y += list(df_p['state'].values)
    if abs(y[-1]-6)<0.2:
        y += list(6*np.ones(9-len(y)))
    else:
        y += list(7*np.ones(9-len(y)))
    all_y.append(np.array(y).reshape(1,-1))

In [ ]:
outcomes = np.concatenate(all_y, axis=0)

In [ ]:
outcomes = outcomes.astype(int)

In [ ]:
import plotly.graph_objects as go
#group2name = { i: name for name, i in possible_response.items()}
possible_response2 = {'relapse free': 1, 'complete response': 2, "partial response": 3,
                     "progressive disease": 5, 'stable disease': 4, "died": 6}

group2name = { i: name for name, i in possible_response2.items()}

group2name[7] = 'out of study'

x = []
for i in range(1,11):
    x.append(i*3)
    
# Function to get colors from a colormap
def get_colors_from_colormap(num_colors, colormap_name='coolwarm'):
    colormap = plt.get_cmap(colormap_name)
    colors = [colormap(i) for i in np.linspace(0, 1, num_colors)]
    return colors

import numpy as np
colors = ['#8c510a', '#d8b365', '#f6e8c3', '#c7eae5', '#5ab4ac', '#01665e'][::-1]
colors.append("gray")

fig = go.Figure()
for igp, gp in enumerate([i for i in range(1,8)]):
    y = np.mean(outcomes==gp, axis=0)
    fig.add_bar(x=x,y=y, name=group2name[gp],marker=dict(
        color=colors[igp] # Set custom colors
    ))
fig.update_layout(
    barmode="relative",
    xaxis_title="Time in months",
    yaxis_title="Proportions",
    font=dict(size=18),  # Increase font size
    xaxis=dict(
        tickmode='array',
        tickvals=x,
        ticktext=[str(val) for val in x]  # Use exact values in `x` for ticks
    )
)
#plt.title('Dimension = '+str(i_d+1))

fig.show()

# Different treatments

In [ ]:
sample_names_df_final = np.unique(df_final['tupro_id'])
all_y = []
for sample in sample_names_df_final:
    df_p = df_final.loc[df_final['tupro_id']==sample]
    df_p = df_p.sort_values(['days']).reset_index(drop=True)
    y = []
    for index, row in df_p.iterrows():
        y +=  [[row['immunotherapies_tot_prev_state']>0,row['chemotherapies_tot_prev_state']>0,row['targeted_tot_prev_state']>0]]
    y += [[-1,-1,-1] for i in range((9-len(y)))]
    all_y.append(np.array(y)[None,:,:])

In [ ]:
outcomes = np.concatenate(all_y, axis=0)

In [ ]:
outcomes = outcomes.astype(int)

In [ ]:
import plotly.graph_objects as go
group2name = { (1,0,0): 'I', (0,1,0): 'C', (0,0,1): 'T', (1,1,0):'I+C', (1,0,1):'I+T', (0,1,1):'C+T', (1,1,1):'I+C+T', (0,0,0):'No treatment', (-1,-1,-1):'No data'}



x = []
for i in range(1,10):
    x.append(str((i-1)*3)+'-'+str(i*3))
    
# Function to get colors from a colormap
def get_colors_from_colormap(num_colors, colormap_name='coolwarm'):
    colormap = plt.get_cmap(colormap_name)
    colors = [colormap(i) for i in np.linspace(0, 1, num_colors)]
    return colors

import numpy as np
colors=['green', 'red',  'blue', 'yellow', 'cyan', 'purple', 'white', 'black', 'gray']

fig = go.Figure()
n = len(sample_names_df_final)
for igp, gp in enumerate(list(group2name.keys())):
    y = np.zeros(len(x))
    for i_sample in range(n):
        for j in range(len(x)):
            if (outcomes[i_sample,j,:]==gp).all():
                y[j]+=1/n
    fig.add_bar(x=x,y=y, name=group2name[gp],marker=dict(
        color=colors[igp] # Set custom colors
    ))
fig.update_layout(
    barmode="relative",
    xaxis_title="Time in months",
    yaxis_title="Proportions",
    font=dict(size=18),  # Increase font size
    xaxis=dict(
        tickmode='array',
        tickvals=x,
        ticktext=[str(val) for val in x]  # Use exact values in `x` for ticks
    )
)
#plt.title('Dimension = '+str(i_d+1))
fig.show()

# Visualization of feature of the drug effect (ratio of the number of non malignant cells versus the one of the tumor cells)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sample DataFrame (replace with your actual data)
# df = pd.DataFrame({'tupro_id': [...], 'evolution_effect': [...], 'days': [...]})

# Pivot the data to have 'tupro_id' as rows, 'days' as columns, and 'evolution_effect' as values
pivot_df = df_final.pivot(index='tupro_id', columns='days', values='evolution_effect')

# Plotting
plt.figure(figsize=(12, 8))
sns.heatmap(pivot_df, cmap='coolwarm', cbar_kws={'label': 'Evolution Effect'})
plt.title('Evolution of drug effect over Time for Each Patient')
plt.xlabel('Days')
plt.ylabel('Patients')
plt.savefig('drug_effect.png', dpi=250, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sample DataFrame (replace with your actual data)
# df = pd.DataFrame({'tupro_id': [...], 'evolution_effect': [...], 'days': [...]})


# Step 2: Pivot the data to have 'tupro_id' as rows, 'days' as columns, and normalized 'evolution_effect' as values
pivot_df = df_final.pivot(index='tupro_id', columns='days', values='evolution_effect_normalized')

# Step 3: Plotting the heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(pivot_df, cmap='coolwarm', cbar_kws={'label': 'Normalized Evolution Effect'})
plt.title('Normalized Evolution of drug effect over Time for Each Patient')
plt.xlabel('Days')
plt.ylabel('Patients')
plt.savefig('normalized_drug_effect.png', dpi=250, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sample DataFrame (replace with your actual data)
# df = pd.DataFrame({'tupro_id': [...], 'evolution_effect': [...], 'days': [...]})

# Pivot the data to have 'tupro_id' as rows, 'days' as columns, and 'evolution_effect' as values
pivot_df = df_final.pivot(index='tupro_id', columns='days', values='evolution_score')

# Plotting
plt.figure(figsize=(12, 8))
sns.heatmap(pivot_df, cmap='coolwarm', cbar_kws={'label': 'Evolution Effect'})
plt.title('Evolution of drug score over Time for Each Patient')
plt.xlabel('Days')
plt.ylabel('Patients')
plt.savefig('drug_score.png', dpi=250, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sample DataFrame (replace with your actual data)
# df = pd.DataFrame({'tupro_id': [...], 'evolution_effect': [...], 'days': [...]})


# Step 2: Pivot the data to have 'tupro_id' as rows, 'days' as columns, and normalized 'evolution_effect' as values
pivot_df = df_final.pivot(index='tupro_id', columns='days', values='evolution_score_normalized')

# Step 3: Plotting the heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(pivot_df, cmap='coolwarm', cbar_kws={'label': 'Normalized Evolution Effect'})
plt.title('Normalized Evolution of drug effect over Time for Each Patient')
plt.xlabel('Days')
plt.ylabel('Patients')
plt.savefig('normalized_drug_score.png', dpi=250, bbox_inches='tight')
plt.show()

# Save

In [ ]:
folder = './R_results/save/'

In [ ]:
ite = 3
dftest = pd.read_csv(folder+'baseline/testData{0}.csv'.format(ite))
pred_probas = pd.read_csv(folder+'baseline/pred_probs{0}.csv'.format(ite))

In [ ]:
samples_test = np.unique(dftest['tupro_id'].values)

In [ ]:
dftest

In [ ]:
pred_probas

In [ ]:
probs_p = pred_probas.loc[pred_probas['x']==4]
probs_p

In [ ]:
ystates

In [ ]:
count = 1
for i, sample in enumerate(samples_test):
    df_p = dftest.loc[dftest['tupro_id']==sample]
    df_p = df_p.sort_values(['days']).reset_index(drop=True)
    ystates_temp = [df_p.iloc[0]['prev_state']]
    ystates_temp += list(df_p['state'].values)
    #plt.scatter([3*i for i in range(len(y))], y)
    
    group2name = { 1: 'Relapse free / CR', 2: 'PR', 3: 'SD', 4: 'PD', 5: 'Death'}
    
    # convert
    ystates = []
    for el in ystates_temp:
        if el=='1_2':
            ystates.append(1)
        else:
            ystates.append(int(el)-1)
    

    x = []
    for i in range(1,len(df_p)+2):
        x.append(str(i*3))

    # Function to get colors from a colormap
    def get_colors_from_colormap(num_colors, colormap_name='coolwarm'):
        colormap = plt.get_cmap(colormap_name)
        colors = [colormap(i) for i in np.linspace(0, 1, num_colors)]
        return colors

    from mycolorpy import colorlist as mcp
    import numpy as np
    colors=['cyan',  'blue', 'orange', 'pink', 'red', 'gray']
    #colors=mcp.gen_color(cmap="winter",n=len(group2name))

    fig = go.Figure()
    
    yscatter = [0]
    scatter_cols = ['gray']
    for x_idx in range(count,count+len(df_p)):
        probs_p = pred_probas.loc[pred_probas['x']==x_idx]['Mean']
        probs_p = [0] + list(probs_p)
        probs_p = np.cumsum(probs_p)
        yscatter.append(0.5* (probs_p[ystates[x_idx-count+1]-1] + probs_p[ystates[x_idx-count+1]]))
        scatter_cols.append(colors[ystates[x_idx-count]])
        if x_idx==count:
            yscatter[0] = 0.5* (probs_p[ystates[x_idx-count]-1] + probs_p[ystates[x_idx-count]])

    for igp in range(1,6):
        y = [0]
        for x_idx in range(count,count+len(df_p)):
            probs_p = pred_probas.loc[pred_probas['x']==x_idx]['Mean'].values
            y.append(probs_p[igp-1])
        fig.add_bar(x=x,y=y, name=group2name[igp],marker=dict(
            color=colors[igp-1] # Set custom colors
        ))
        
    fig.add_scatter(x=x, y=yscatter, mode='markers', name="Observed status", marker=dict(
        color='black',  # You can customize the scatter plot color
        size=10,  # You can customize the marker size
        symbol='circle'  # You can customize the marker symbol
    ))
    
    fig.update_layout(
        barmode="relative",
        xaxis_title="Time in months",
        yaxis_title="Proportions",
        font=dict(size=18),  # Increase font size
        xaxis=dict(
            tickmode='array',
            tickvals=x,
            ticktext=[str(val) for val in x]  # Use exact values in `x` for ticks
        )
    )
    
    
    # plt.title('Patient: {0}'.format(sample))
    fig.show()
    
    
    count += len(df_p)

In [ ]:
import plotly.graph_objects as go
group2name = { (1,0,0): 'I', (0,1,0): 'C', (0,0,1): 'T', (1,1,0):'I+C', (1,0,1):'I+T', (0,1,1):'C+T', (1,1,1):'I+C+T', (0,0,0):'No treatment', (-1,-1,-1):'No data'}



x = []
for i in range(1,10):
    x.append(str((i-1)*3)+'-'+str(i*3))
    
# Function to get colors from a colormap
def get_colors_from_colormap(num_colors, colormap_name='coolwarm'):
    colormap = plt.get_cmap(colormap_name)
    colors = [colormap(i) for i in np.linspace(0, 1, num_colors)]
    return colors

from mycolorpy import colorlist as mcp
import numpy as np
colors=['green', 'red',  'blue', 'yellow', 'cyan', 'purple', 'white', 'black', 'gray']

fig = go.Figure()
n = len(sample_names_df_final)
for igp, gp in enumerate(list(group2name.keys())):
    y = np.zeros(len(x))
    for i_sample in range(n):
        for j in range(len(x)):
            if (outcomes[i_sample,j,:]==gp).all():
                y[j]+=1/n
    fig.add_bar(x=x,y=y, name=group2name[gp],marker=dict(
        color=colors[igp] # Set custom colors
    ))
fig.update_layout(
    barmode="relative",
    xaxis_title="Time in months",
    yaxis_title="Proportions",
    font=dict(size=18),  # Increase font size
    xaxis=dict(
        tickmode='array',
        tickvals=x,
        ticktext=[str(val) for val in x]  # Use exact values in `x` for ticks
    )
)
#plt.title('Dimension = '+str(i_d+1))
fig.show()

In [ ]:
df_final.loc[df_final]

In [ ]:
plt.plot(np.sort(df_final['pi0']))
plt.plot(np.sort(df_final['min_pit']))
plt.plot(np.sort(df_final['max_pit']))

In [ ]:
features_final

# Appendix

In [ ]:
count = 0
for i, sample in enumerate(df.index):
    print(sample)
    print(df.iloc[i]['biopsy_date'], df.iloc[i]['treatment_after'])
    print('\n')
    valid_patient = True
    for drug in missing_drugs:
        if drug in df.iloc[i]['treatment_after']:
            valid_patient = False
    if valid_patient:
        count += 1
print('Number of patient with treatment in the Fast Drug data: ', count)